# Faruq-v3 — Unified private Kaggle core updater

Membangun ulang private Kaggle Dataset `faruq-v3-experiment-core-v1` agar sekaligus memuat kontrak AF2-spectral dan top-controls.

Notebook ini **tidak menjalankan training** dan **tidak memasukkan test**. Sumber artefak berasal dari folder proyek Drive `Coffee_Bean_Detection`.

Sebelum Run all, buat Colab secrets dan aktifkan notebook access: `KAGGLE_USERNAME` dan `KAGGLE_API_TOKEN`.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=True)

import importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path

BRANCH='codex/af2-ffab2-from-start-dct'
REPO=Path('/content/coffee-bean-detection')
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)

subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','ultralytics==8.4.96','kaggle==2.2.2'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('BRANCH:',BRANCH)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('ULTRALYTICS:',__import__('ultralytics').__version__)


In [ ]:
from coffee_detector.drive_project import resolve_drive_project_root
from coffee_detector.experiments.prepare_af2_spectral_kaggle import (
    PROJECT_ARTIFACTS as AF2_PROJECT_ARTIFACTS,
    build_af2_spectral_kaggle_bundle,
)
from coffee_detector.experiments.prepare_top_controls_kaggle import (
    ARTIFACTS as TOP_PROJECT_ARTIFACTS,
    build_top_controls_canonical_kaggle_core,
)

required=sorted(set(AF2_PROJECT_ARTIFACTS.values()) | set(TOP_PROJECT_ARTIFACTS.values()) | {'bundles/faruq-development-v3-grouped.tar'})
print('REQUIRED PROJECT ARTIFACTS:')
for item in required: print(' -',item)

PROJECT=resolve_drive_project_root(required_relative_paths=tuple(required))
print('PROJECT ROOT:',PROJECT)


In [ ]:
BUNDLE=Path('/content/faruq-v3-experiment-core-v1')
if BUNDLE.exists(): shutil.rmtree(BUNDLE)
BUNDLE.mkdir(parents=True)

print('BUILD AF2-SPECTRAL CORE...')
af2_manifest=build_af2_spectral_kaggle_bundle(PROJECT,BUNDLE)
print('AF2 MANIFEST:',af2_manifest['manifest'])

print('BUILD TOP-CONTROLS CORE...')
top_core=build_top_controls_canonical_kaggle_core(PROJECT,BUNDLE)
print('TOP CORE MANIFEST:',top_core['manifest'])

required_files=(
    'faruq-development-v3-grouped.tar.bin',
    'af2_spectral_kaggle_manifest.json',
    'D0_seed42_best.pt','D0_seed123_best.pt','D0_seed2026_best.pt',
    'top_controls_kaggle_manifest.json','top_controls_canonical_core_manifest.json',
    'STB1_seed123_best.pt','STB1_seed2026_best.pt',
    'AF2_seed123_best.pt','AF2_seed2026_best.pt',
)
missing=[name for name in required_files if not (BUNDLE/name).is_file()]
if missing: raise RuntimeError(f'Unified bundle belum lengkap: {missing}')

print('\nUNIFIED BUILD PASS')
for name in required_files:
    p=BUNDLE/name
    print(f'{name}: {p.stat().st_size:,} bytes')


In [ ]:
username=userdata.get('KAGGLE_USERNAME')
token=userdata.get('KAGGLE_API_TOKEN')
assert username and token,'Aktifkan secrets KAGGLE_USERNAME dan KAGGLE_API_TOKEN.'
username=username.strip()
assert username and '/' not in username,'KAGGLE_USERNAME tidak valid.'
os.environ['KAGGLE_USERNAME']=username
os.environ['KAGGLE_API_TOKEN']=token
os.environ['KAGGLE_KEY']=token

dataset_id=f'{username}/faruq-v3-experiment-core-v1'
metadata={
    'title':'Faruq V3 Experiment Core V1',
    'id':dataset_id,
    'licenses':[{'name':'other'}],
    'isPrivate':True,
}
(BUNDLE/'dataset-metadata.json').write_text(json.dumps(metadata,indent=2)+'\n',encoding='utf-8')

probe=subprocess.run(['kaggle','datasets','list','--mine','--search','faruq-v3-experiment-core-v1','--csv'],text=True,capture_output=True)
if probe.returncode!=0:
    print(probe.stdout); print(probe.stderr)
    raise RuntimeError('Autentikasi/probe Kaggle gagal sebelum upload.')
dataset_exists=dataset_id.lower() in probe.stdout.lower()
message='Unified Faruq-v3 core: AF2 spectral D0 bundle plus top-controls bundle'
if dataset_exists:
    print('Dataset existing ditemukan. Upload VERSION baru...',flush=True)
    upload=subprocess.run(['kaggle','datasets','version','-p',str(BUNDLE),'-m',message,'--keep-tabular'],check=False)
    action='UPDATED'
else:
    print('Dataset belum ada. CREATE...',flush=True)
    upload=subprocess.run(['kaggle','datasets','create','-p',str(BUNDLE),'--keep-tabular'],check=False)
    action='CREATED'
if upload.returncode!=0: raise RuntimeError(f'Kaggle {action.lower()} gagal: returncode={upload.returncode}')
print('UPLOAD:',action)
print('DATASET:',dataset_id)
print('PRIVATE: True | TEST: False')


In [ ]:
required_remote=(
    'faruq-development-v3-grouped.tar.bin',
    'af2_spectral_kaggle_manifest.json',
    'D0_seed42_best.pt','D0_seed123_best.pt','D0_seed2026_best.pt',
    'top_controls_kaggle_manifest.json','top_controls_canonical_core_manifest.json',
    'STB1_seed123_best.pt','STB1_seed2026_best.pt',
    'AF2_seed123_best.pt','AF2_seed2026_best.pt',
)
listing=''
for attempt in range(18):
    check=subprocess.run(['kaggle','datasets','files',dataset_id,'--page-size','100','--csv'],text=True,capture_output=True)
    listing=check.stdout
    if check.returncode==0 and all(name in listing for name in required_remote): break
    print(f'Menunggu indeks Kaggle: {attempt+1}/18',flush=True)
    time.sleep(10)
else:
    print(listing); print(check.stderr)
    raise RuntimeError('Upload diterima tetapi unified core belum lengkap di indeks Kaggle.')

print('\nVERIFIKASI KAGGLE: PASS')
for name in required_remote: print('OK:',name)
print('\nSekarang kembali ke notebook Kaggle AF2 vs AF2+FFAB2, remove input lama lalu Add Input versi terbaru faruq-v3-experiment-core-v1, kemudian Run All.')
